# Network Anomaly Detection: GNN vs DBSCAN
## Interactive Demo

In [ ]:
import sys
sys.path.append('..')
from utils import data_loader, feature_generator, dataset, baseline, models, train, visualization
import config
import torch
import numpy as np
from sklearn.metrics import precision_recall_curve

# 1. Load

In [ ]:
G_clean = data_loader.load_network()
G_clean, G_dirty, df_nodes, df_edges_train, df_edges_test = feature_generator.generate_features_and_anomalies(G_clean)

# 2. Baseline

In [ ]:
metrics = baseline.run_dbscan(df_nodes)
print(metrics)

# 3. Train

In [ ]:
train_data = dataset.create_pyg_data(df_nodes, df_edges_train)
model = models.build_model(train_data.num_features)
model, _ = train.train_model(model, train_data)

# 4. Eval

In [ ]:
test_data = dataset.create_pyg_data(df_nodes, df_edges_test)
model.eval()
with torch.no_grad():
    z = model.encode(test_data.x, test_data.edge_index)
    adj_pred = model.decoder.forward_all(z)

node_scores = []
for i in range(len(df_nodes)):
    neighbors = list(G_dirty.neighbors(i))
    if not neighbors:
        node_scores.append(0)
        continue
    link_probs = [adj_pred[i, n].item() for n in neighbors]
    node_scores.append(1.0 - min(link_probs))

# 5. Viz

In [ ]:
visualization.plot_network(G_dirty, adj_pred, 0.5)